# TIỀN XỬ LÝ DỮ LIỆU VÀ KỸ THUẬT ĐẶC TRƯNG
## Đề tài: Dự đoán nhu cầu xe đạp công cộng (UCI Bike Sharing Dataset)

Sổ tay này thực hiện các bước làm sạch dữ liệu, xử lý giá trị bất thường, loại bỏ các biến gây rò rỉ dữ liệu và đa cộng tuyến, đồng thời chuẩn bị dữ liệu đầu vào tối ưu cho các mô hình học máy cổ điển.

# 1. Khởi tạo môi trường và nạp dữ liệu

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Đọc tập dữ liệu theo giờ gốc
df = pd.read_csv('../data/hour.csv')
print(f"Kích thước tập dữ liệu gốc: {df.shape[0]} dòng, {df.shape[1]} cột")
df.head()

Kích thước tập dữ liệu gốc: 17379 dòng, 17 cột


# 2. Loại bỏ các đặc trưng định danh và chống rò rỉ dữ liệu (Data Leakage)

Căn cứ trên kết quả phân tích khám phá dữ liệu (EDA):
- **`instant` và `dteday`:** Là các chỉ số thứ tự và ngày định danh chuỗi, không mang giá trị quy luật khái quát hóa cho mô hình nên cần được loại bỏ.
- **`casual` và `registered`:** Là hai thành phần cấu thành trực tiếp biến mục tiêu ($cnt = casual + registered$). Nếu giữ lại trong tập đặc trưng $X$, mô hình sẽ bị hiện tượng **Rò rỉ dữ liệu (Data Leakage)** nghiêm trọng, thuật toán chỉ việc học phép cộng giữa hai cột này để đạt độ chính xác giả tạo $100\%$ nhưng hoàn toàn vô dụng trên thực tế khi chưa biết trước lượng khách thuê.

In [2]:
# Loại bỏ các cột định danh và các cột gây rò rỉ dữ liệu
drop_leakage_cols = ['instant', 'dteday', 'casual', 'registered']
df = df.drop(columns=drop_leakage_cols)

print(f"Đã loại bỏ các cột: {drop_leakage_cols}")
print(f"Kích thước dữ liệu hiện tại: {df.shape}")

Đã loại bỏ các cột: ['instant', 'dteday', 'casual', 'registered']
Kích thước dữ liệu hiện tại: (17379, 13)


# 3. Loại bỏ đặc trưng đa cộng tuyến (Multicollinearity)

Biểu đồ ma trận tương quan ở bước EDA đã chứng minh hệ số tương quan giữa `temp` (nhiệt độ thực tế) và `atemp` (nhiệt độ cảm nhận) đạt mức gần như tuyệt đối ($r = 0.9877 \approx 0.99$).

Trong hồi quy tuyến tính cổ điển, hiện tượng đa cộng tuyến hoàn hảo làm ma trận $(X^T X)$ gần suy biến, khiến phương sai của các trọng số hồi quy bùng nổ vô hạn ($\text{Var}(\hat{\beta}) \to \infty$), làm mất tính ổn định và khả năng diễn giải của mô hình. Do đó, chúng tôi giữ lại biến vật lý khách quan `temp` và loại bỏ biến cảm tính chủ quan `atemp`.

In [3]:
# Loại bỏ cột atemp để triệt tiêu hiện tượng đa cộng tuyến
df = df.drop(columns=['atemp'])

print("Đã loại bỏ cột 'atemp'.")
print(f"Kích thước dữ liệu hiện tại: {df.shape}")

Đã loại bỏ cột 'atemp'.
Kích thước dữ liệu hiện tại: (17379, 12)


# 4. Xử lý giá trị bất hợp lý của độ ẩm (hum == 0)

Qua kiểm toán thống kê mô tả, chúng tôi phát hiện 22 bản ghi có độ ẩm $hum = 0$ ($0\%$) diễn ra liên tục trong ngày 10/03/2011 do sự cố gián đoạn tín hiệu cảm biến thời tiết.

Để bảo toàn tính liên tục của chuỗi thời gian 17,379 giờ, chúng tôi áp dụng phương pháp nội suy thay thế (Imputation) bằng cách gán giá trị **trung vị (Median)** của độ ẩm theo từng tháng và từng khung giờ tương ứng từ các quan sát hợp lệ ($hum > 0$).

In [4]:
# Kiểm tra số lượng bản ghi hum == 0 trước khi xử lý
zero_hum_count = (df['hum'] == 0).sum()
print(f"Số dòng có độ ẩm hum == 0 trước khi xử lý: {zero_hum_count}")

# Tính giá trị trung vị độ ẩm theo (tháng, giờ) cho các bản ghi hợp lệ
valid_hum = df[df['hum'] > 0]
median_hum_by_mnth_hr = valid_hum.groupby(['mnth', 'hr'])['hum'].median()

# Thay thế các giá trị 0 bằng trung vị theo tháng và khung giờ tương ứng
df['hum'] = df.apply(
    lambda row: median_hum_by_mnth_hr.get((row['mnth'], row['hr']), 0.59) if row['hum'] == 0 else row['hum'],
    axis=1
)

# Kiểm tra lại sau xử lý
zero_hum_after = (df['hum'] == 0).sum()
print(f"Số dòng có độ ẩm hum == 0 sau khi xử lý: {zero_hum_after}")
print(f"Giá trị độ ẩm nhỏ nhất hiện tại: {df['hum'].min():.4f}")

Số dòng có độ ẩm hum == 0 trước khi xử lý: 22
Số dòng có độ ẩm hum == 0 sau khi xử lý: 0
Giá trị độ ẩm nhỏ nhất hiện tại: 0.0800


# 5. Kiểm tra thông tin tập dữ liệu sau bước làm sạch cơ bản

Tập dữ liệu hiện tại còn đúng 12 thuộc tính (gồm 11 biến độc lập sạch và 1 biến mục tiêu `cnt`), sẵn sàng cho các bước lọc ngoại lai, tạo đặc trưng chu kỳ và chuẩn hóa tiếp theo.

In [5]:
# Kiểm tra tổng quan kiểu dữ liệu và giá trị khuyết
print("=== THÔNG TIN DỮ LIỆU HIỆN TẠI ===")
df.info()
print("\n=== THỐNG KÊ MÔ TẢ CÁC BIẾN SỐ HỌC ===")
df[['temp', 'hum', 'windspeed', 'cnt']].describe().T

=== THÔNG TIN DỮ LIỆU HIỆN TẠI ===
<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   season      17379 non-null  int64  
 1   yr          17379 non-null  int64  
 2   mnth        17379 non-null  int64  
 3   hr          17379 non-null  int64  
 4   holiday     17379 non-null  int64  
 5   weekday     17379 non-null  int64  
 6   workingday  17379 non-null  int64  
 7   weathersit  17379 non-null  int64  
 8   temp        17379 non-null  float64
 9   hum         17379 non-null  float64
 10  windspeed   17379 non-null  float64
 11  cnt         17379 non-null  int64  
dtypes: float64(3), int64(9)
memory usage: 1.6 MB

=== THỐNG KÊ MÔ TẢ CÁC BIẾN SỐ HỌC ===
